Install pyvips to tile images. This is needed to get around the memory limitation within the Kaggle environment.

In [ ]:
!ls /kaggle/input/pyvips-python-and-deb-package-gpu
# intall the deb packages
!yes | dpkg -i --force-depends /kaggle/input/pyvips-python-and-deb-package-gpu/linux_packages/archives/*.deb
# install the python wrapper
!pip install pyvips -f /kaggle/input/pyvips-python-and-deb-package-gpu/python_packages/ --no-index

In [ ]:
%load /kaggle/input/ubc-ocean-src/tiles.py

In [1]:
%run /kaggle/input/ubc-ocean-src/model.py

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
import torch
import pandas as pd
from torch.utils.data import DataLoader
from torchvision import transforms
from pathlib import Path

def prepare_ucbocean_test_dataset(raw_dataset, data_dir, image_resize_max_edge_size):
    '''Prepare the UBC-OCEAN dataset for a custom data loader.
    raw_dataset: The raw dataset loaded from the csv file as a pandas dataframe
    returns: A PyTorch dataset
    '''
    composed = transforms.Compose([
        RescaleWithAspectRatioTransform(image_resize_max_edge_size),
        transforms.ToTensor(), # Scale the each tile to a numpy.ndarray (H x W x C) in the range [0, 255] to a torch.FloatTensor of shape (C x H x W) in the range [0.0, 1.0]
    ])
    dataset = UCBOCEANTestDataset(raw_dataset, data_dir, transform=composed)
    return dataset

TEST_DATA_DIR = Path('/kaggle/input/UBC-OCEAN/test_images')
SUBMISSION_FILE = Path('/kaggle/working/submission.csv')

# Load the test data from the csv file
df_test = pd.read_csv('/kaggle/input/UBC-OCEAN/test.csv')
df_train = pd.read_csv('/kaggle/input/UBC-OCEAN/train.csv')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize model instance
label_count = 5
model = UBCOCEANNet1(label_count)  # replace with your model and its parameters
model.to(device)

# Load checkpoint
checkpoint_path = Path('/kaggle/input/ubc-ocean-checkpoints/checkpoint_1698955439.1921322_9.pth.tar')
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

# Prepare DataLoader for your test data
image_resize_max_edge_size = 2048
dataset = prepare_ucbocean_test_dataset(df_test, TEST_DATA_DIR,  image_resize_max_edge_size)
test_data_loader = DataLoader(dataset,  batch_size=1, shuffle=False, num_workers=0)

# List to hold the predictions
labels = []
image_ids = []

# Set the model to evaluation mode
model.eval()
with torch.no_grad():
    for data in test_data_loader:
        # Assuming that your DataLoader returns a tuple of features and labels
        inputs, image_id = data  # We do not need labels for inference, but we still need the associated image ID
        inputs = inputs.to(device)
        
        # Forward pass
        outputs = model(inputs)
        
        # Get predictions
        _, predicted = torch.max(outputs.data, 1)
        
        # Move the predictions to CPU and collect them
        labels.extend(predicted.cpu().numpy())
        image_ids.extend(image_id.numpy())

# Write the predictions to a submission file
df_submission = pd.DataFrame({
    'image_id': image_ids,
    'label': labels
})
df_submission['label'] = OrdinalEncodeTransform(df_train).decode(df_submission['label']) # Transform the predictions back to their original labels
df_submission.to_csv(SUBMISSION_FILE, index=False)


/opt/conda/lib/python3.10/site-packages/torch/cuda/__init__.py:173: UserWarning: 
NVIDIA GeForce RTX 3080 with CUDA capability sm_86 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_37 sm_60 sm_70 sm_75 compute_70 compute_75.
If you want to use the NVIDIA GeForce RTX 3080 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(incompatible_device_warn.format(device_name, capability, " ".join(arch_list), device_name))
